# 2. Tracking Algorithms
Goal: go from a single-object tracker to a full multi-object tracker (detect → predict → assign → update).

## 2.1 Setup

In [ ]:
import cv2
import numpy as np
from scipy.spatial import distance as dist
from scipy.optimize import linear_sum_assignment

## 2.2 Single-object tracker (baseline)
OpenCV's built-in trackers follow ONE box. Good starting point, not enough for many cells/people.

In [ ]:
tracker = cv2.legacy.TrackerCSRT_create()  # needs opencv-contrib-python

AttributeError: module 'cv2' has no attribute 'legacy'

In [ ]:
cap = cv2.VideoCapture("assets/CellVideo.avi")
ret, frame = cap.read()
bbox = (100, 100, 20, 20)  # (x, y, w, h) — or use cv2.selectROI(frame)
tracker.init(frame, bbox)

In [ ]:
ret, frame = cap.read()
ok, bbox = tracker.update(frame)
print(ok, bbox)

Limitation: 1 tracker = 1 object, and it can't add/remove objects on its own. We need a multi-object approach.

## 2.3 Centroid tracking (nearest-neighbor ID assignment)

In [ ]:
next_id = 0
tracked_objects = {}  # id -> centroid

In [ ]:
def update_centroid_tracker(tracked_objects, next_id, new_centroids, max_dist=50):
    if len(tracked_objects) == 0:
        for c in new_centroids:
            tracked_objects[next_id] = c
            next_id += 1
        return tracked_objects, next_id

    ids = list(tracked_objects.keys())
    old_c = list(tracked_objects.values())
    D = dist.cdist(old_c, new_centroids)

    rows = D.min(axis=1).argsort()
    cols = D.argmin(axis=1)[rows]

    used_rows, used_cols = set(), set()
    new_tracked = {}
    for r, c in zip(rows, cols):
        if r in used_rows or c in used_cols or D[r, c] > max_dist:
            continue
        new_tracked[ids[r]] = new_centroids[c]
        used_rows.add(r); used_cols.add(c)

    for c_idx, c in enumerate(new_centroids):
        if c_idx not in used_cols:
            new_tracked[next_id] = c
            next_id += 1

    return new_tracked, next_id

Why it works: the nearest old centroid is usually the same object, as long as it didn't move far between frames.

## 2.4 Kalman filter — predicting motion
Mirrors `configureKalmanFilter` + `predict`/`correct` in `trackCellMotion.mlx`.

In [ ]:
def create_kalman(x, y):
    kf = cv2.KalmanFilter(4, 2)
    kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
    kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    kf.statePre = np.array([x, y, 0, 0], np.float32)
    return kf

In [ ]:
kf = create_kalman(100, 100)
pred = kf.predict()
print("predicted:", pred[:2].flatten())

In [ ]:
measurement = np.array([102, 98], np.float32)
kf.correct(measurement)

`predict()` = where we expect the object next. `correct()` = update the estimate with the real detection.

## 2.5 Assignment with the Hungarian algorithm
Mirrors MATLAB's `assignDetectionsToTracks` — same cost-matrix idea, optimal instead of greedy.

In [ ]:
def assign_detections_to_tracks(track_preds, detections, cost_threshold=50):
    cost = dist.cdist(track_preds, detections)
    row_idx, col_idx = linear_sum_assignment(cost)

    matches, unmatched_tracks, unmatched_dets = [], [], []
    for r in range(len(track_preds)):
        if r not in row_idx:
            unmatched_tracks.append(r)
    for c in range(len(detections)):
        if c not in col_idx:
            unmatched_dets.append(c)
    for r, c in zip(row_idx, col_idx):
        if cost[r, c] > cost_threshold:
            unmatched_tracks.append(r); unmatched_dets.append(c)
        else:
            matches.append((r, c))
    return matches, unmatched_tracks, unmatched_dets

## 2.6 Combine into a multi-object tracker
Same 4 steps as the MATLAB main loop: predict → assign → update matched → create/remove tracks.

In [ ]:
class Track:
    def __init__(self, track_id, centroid):
        self.id = track_id
        self.kf = create_kalman(*centroid)
        self.missed = 0

In [ ]:
class MultiObjectTracker:
    def __init__(self, max_missed=5, cost_threshold=50):
        self.tracks = {}
        self.next_id = 0
        self.max_missed = max_missed
        self.cost_threshold = cost_threshold

    def predict_all(self):
        preds = {}
        for tid, t in self.tracks.items():
            p = t.kf.predict()
            preds[tid] = (p[0, 0], p[1, 0])
        return preds

    def update(self, detections):
        preds = self.predict_all()
        ids = list(preds.keys())

        if ids and detections:
            matches, unmatched_tracks, unmatched_dets = assign_detections_to_tracks(
                list(preds.values()), detections, self.cost_threshold)
        else:
            matches, unmatched_tracks = [], list(range(len(ids)))
            unmatched_dets = list(range(len(detections)))

        for r, c in matches:
            tid = ids[r]
            self.tracks[tid].kf.correct(np.array(detections[c], np.float32))
            self.tracks[tid].missed = 0

        for r in unmatched_tracks:
            self.tracks[ids[r]].missed += 1

        for c in unmatched_dets:
            self.tracks[self.next_id] = Track(self.next_id, detections[c])
            self.next_id += 1

        self.tracks = {tid: t for tid, t in self.tracks.items() if t.missed <= self.max_missed}
        return {tid: t.kf.statePost[:2].flatten() for tid, t in self.tracks.items()}

## 2.7 Quick test

In [ ]:
mot = MultiObjectTracker()
print(mot.update([(120, 80), (300, 200)]))

In [ ]:
print(mot.update([(125, 82), (310, 205), (50, 50)]))  # 2 continue, 1 new